# Sales & Demand Forecasting
### Future Interns – ML Task 1
**Dataset:** Superstore Sales Dataset  
**Goal:** Predict future monthly sales using historical data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

plt.rcParams['figure.figsize'] = (12, 4)

## Step 1 – Load & Clean Data

In [ ]:
df = pd.read_csv('Sample_-_Superstore.csv', encoding='latin1')
print('Shape:', df.shape)
df.head()

In [ ]:
# Check missing values
df.isnull().sum()

In [ ]:
df.drop_duplicates(inplace=True)
df.dropna(subset=['Sales'], inplace=True)

df['Order Date'] = pd.to_datetime(df['Order Date'], dayfirst=True)

# Extract time features
df['Year']  = df['Order Date'].dt.year
df['Month'] = df['Order Date'].dt.month
df['Day']   = df['Order Date'].dt.day

print('Date range:', df['Order Date'].min(), 'to', df['Order Date'].max())
df[['Order Date','Year','Month','Day','Sales']].head()

## Step 2 – Aggregate Monthly Sales

In [ ]:
monthly = (
    df.groupby(['Year', 'Month'])['Sales']
    .sum()
    .reset_index()
)
monthly['Date'] = pd.to_datetime(monthly[['Year', 'Month']].assign(Day=1))
monthly.sort_values('Date', inplace=True)
monthly.reset_index(drop=True, inplace=True)

monthly.head(10)

In [ ]:
plt.plot(monthly['Date'], monthly['Sales'], color='steelblue', linewidth=1.8)
plt.fill_between(monthly['Date'], monthly['Sales'], alpha=0.15, color='steelblue')
plt.title('Monthly Sales Over Time')
plt.xlabel('Date')
plt.ylabel('Sales ($)')
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

## Step 3 – Feature Engineering

In [ ]:
monthly['Month_Num'] = range(len(monthly))   # linear trend
monthly['Sin_Month'] = np.sin(2 * np.pi * monthly['Month'] / 12)  # seasonality
monthly['Cos_Month'] = np.cos(2 * np.pi * monthly['Month'] / 12)

monthly[['Date', 'Sales', 'Month_Num', 'Sin_Month', 'Cos_Month']].head()

## Step 4 – Train Random Forest Model

In [ ]:
features = ['Month_Num', 'Year', 'Month', 'Sin_Month', 'Cos_Month']
X = monthly[features]
y = monthly['Sales']

model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X, y)

train_preds = model.predict(X)

mae  = mean_absolute_error(y, train_preds)
rmse = np.sqrt(mean_squared_error(y, train_preds))

print(f'MAE  : ${mae:,.2f}')
print(f'RMSE : ${rmse:,.2f}')

In [ ]:
plt.plot(monthly['Date'], y, label='Actual Sales', color='steelblue', linewidth=1.8)
plt.plot(monthly['Date'], train_preds, label='Predicted Sales', color='orange', linestyle='--', linewidth=1.8)
plt.title('Actual vs Predicted Monthly Sales')
plt.xlabel('Date')
plt.ylabel('Sales ($)')
plt.legend()
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

## Step 5 – Forecast Future Sales (Next 12 Months)

In [ ]:
months_ahead = 12
last_row     = monthly.iloc[-1]
last_date    = last_row['Date']

future_records = []
for i in range(1, months_ahead + 1):
    future_date = last_date + pd.DateOffset(months=i)
    month_num   = last_row['Month_Num'] + i
    month       = future_date.month
    year        = future_date.year
    future_records.append({
        'Date': future_date,
        'Month_Num': month_num,
        'Year': year,
        'Month': month,
        'Sin_Month': np.sin(2 * np.pi * month / 12),
        'Cos_Month': np.cos(2 * np.pi * month / 12),
    })

future_df = pd.DataFrame(future_records)
future_df['Predicted_Sales'] = model.predict(future_df[features])

future_df[['Date', 'Predicted_Sales']]

In [ ]:
plt.plot(monthly['Date'], monthly['Sales'], label='Historical Sales', color='steelblue', linewidth=1.8)
plt.plot(future_df['Date'], future_df['Predicted_Sales'],
         label='Forecasted Sales', color='green', linestyle='--', linewidth=2, marker='o', markersize=4)
plt.axvline(x=last_date, color='gray', linestyle=':', linewidth=1.2, label='Forecast Start')
plt.title('Sales Forecast – Next 12 Months')
plt.xlabel('Date')
plt.ylabel('Sales ($)')
plt.legend()
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

## Summary

- **Model Used:** Random Forest Regressor
- **Features:** Month number (trend), Year, Month, Sine/Cosine of month (seasonality)
- **Evaluation:** MAE and RMSE on training data
- **Output:** 12-month future sales forecast

### How a Business Can Use This
- Plan inventory before high-sales months
- Allocate staff based on expected demand
- Set realistic revenue targets for each quarter
- Identify slow months and plan promotions in advance